# Image Classification using Transfer Learning

We will classify images of cats and dogs using transfer learning models such as Inception, ResNet and VGGNet.

By end of the this exercise,users will practice:

- Transfer learning, reusing a CNN that was already trained on a large dataset.
- Making predictions on unseen test images and comparing the choice of pretrained model (`inception`, `resnet`, `vgg16`).

In [ ]:
import tensorflow as tf
import tarfile
import urllib.request
import os
import numpy as np

from matplotlib import pyplot as plt
%matplotlib inline

from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications import imagenet_utils

data_path = 'ai-ml-course'
full_dir = os.path.join(data_path, 'transfer-data')
os.makedirs(full_dir, exist_ok=True)

## Dataset

The dataset is hosted publicly by DiRAC on COSMA. **Please make sure you run the cell below to download the data.** 

To access the data from DiRAC Training Academy you need to:

1. Run the first cell,
2. It will create a folder called `ai-ml-course,
3. It will download the necessary files directly into that folder,
4. Run the second cell and wait until the data folder appears.

In [ ]:
base_url = "https://virgodb.cosma.dur.ac.uk/public/diractraining/ai-ml-course/transfer-data/"
files_to_download = ["images.tar.gz", "annotations.tar.gz"]

In [ ]:
for filename in files_to_download:
    path = os.path.join(full_dir, filename)
    folder = os.path.join(full_dir, filename.split('.')[0])      

    if not os.path.isdir(folder):
        if not os.path.exists(path):
            print(f"Downloading {filename} ...")
            urllib.request.urlretrieve(base_url + filename, path)

        print(f"Unpacking {filename} ...")
        with tarfile.open(path) as archive:
            try:
                archive.extractall(full_dir, filter='data')
            except TypeError:                                   
                archive.extractall(full_dir)

print("Data is ready.")

# Add label to each image

This is a data preprocessing step which ensures a proper label is added to each image name. 

In [ ]:
def image_annotations(file_path):
    annotations = {}
    
    with open(file_path, 'r') as f:
        rows = f.read().splitlines()

    for i, row in enumerate(rows):
        image_name, _, _, _ = row.split(' ')
        image_name += '.jpg'
        if image_name[0].lower() != image_name[0]:
            class_name = 'cat'
        else:
            class_name = 'dog'
        annotations[image_name] = class_name
    
    return annotations, i + 1


class_to_index = {'cat': 0, 'dog': 1}
index_to_class = {0: 'cat', 1: 'dog'}

train_path = os.path.join(data_path, 'transfer-data/annotations/trainval.txt')
test_path = os.path.join(data_path, 'transfer-data/annotations/test.txt')

train_annot, count_train = image_annotations(train_path)
test_annot, count_test = image_annotations(test_path)

print('Training examples count:', count_train)
print('Test examples count:', count_test)

# Select images randomly

A function is defined which is used to randomly choose images of given batch size.

In [ ]:
image_dir = os.path.join(data_path, 'transfer-data/images')

def get_random_batch(annot,model_name, batch_size=4):
    all_keys = list(annot.keys())
    total_examples = len(all_keys)
    indices = np.random.choice(range(total_examples), batch_size)
    x = np.zeros((batch_size, 128, 128, 3))
    y = np.zeros((batch_size, 1))
    images = []
    
    for i, index in enumerate(indices):
        image = tf.keras.preprocessing.image.load_img(os.path.join(image_dir, all_keys[index]),
                                                     target_size=(128, 128))
        images.append(image)
        arr = tf.keras.preprocessing.image.img_to_array(image)
        if model_name == 'inception':
            arr = tf.keras.applications.inception_v3.preprocess_input(arr)
        elif model_name == 'vgg16' or model_name == 'resnet':  
            arr = imagenet_utils.preprocess_input(arr)
        arr = np.expand_dims(arr, axis=0)
        x[i] = arr
        y[i] = class_to_index[annot[all_keys[index]]]
    
    return x, y, images

# Load pre-trained model

Here we load a pre-trained model from Keras. The model is specified using the variable model_name. 

In [ ]:
MODELS = {
"inception": InceptionV3,
"resnet": ResNet50,
"vgg16": VGG16 }

model_name = 'inception'
Network = MODELS[model_name]

base_model = Network(weights='imagenet', include_top=False,input_shape=(128, 128, 3), pooling='avg')
base_model.summary()

# Display images

In [ ]:
def display_images(x, y, p, images, index_to_class):
    
    if len(images) < 8:
        print('Need at least 8 examples')
        return None

    plt.figure(figsize=(12, 8))
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i])
        plt.xticks([])
        plt.yticks([])
        gt = int(np.squeeze(y[i]) > 0.5)
        pred = int(np.squeeze(p[i]) > 0.5)
        col = 'green' if gt == pred else 'red'
        plt.xlabel(index_to_class[pred], color=col)
    return plt

x, y, images = get_random_batch(train_annot,model_name, batch_size=8)
display_images(x, y, y, images, index_to_class).show()

# Build and compile model

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        base_model,
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.layers[0].trainable = False
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['acc'])
    return model

model = create_model()
model.summary()

In [ ]:
def data_generator(batch_size, annot):
    while True:
        x, y, _ = get_random_batch(annot,model_name, batch_size)
        yield (x, y)

In [ ]:
batch_size = 32
steps_per_epoch = int(len(list(train_annot.keys()))/batch_size)
validation_steps = int(len(list(test_annot.keys()))/batch_size)

print('Steps per epoch:', steps_per_epoch)
print('Validation steps:', validation_steps)


# Train model

In [ ]:
%%time

H = model.fit(
    data_generator(batch_size, train_annot),
    validation_data=data_generator(batch_size, test_annot),
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    epochs=1
)

# Make predictions

In [ ]:
x, y, images = get_random_batch(test_annot,model_name, batch_size=8)
preds = model.predict(x)
display_images(x, y, preds, images, index_to_class).show()